# Task 7 — Deterministic frequency Stage 1

Extract fixed FFT/DCT/residual fingerprints, compare magnitude-only with bounded phase additions, and keep the early exit disabled. Only `seed_train` and `selection_val` are read.

In [18]:
# 1. Drive and repository. A GPU is not required for this deterministic CPU feature bank.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess
import sys
import shutil

PROJECT_ROOT = Path('/content/cya-techjam26')
REPOSITORY_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if (PROJECT_ROOT / '.git').is_dir():
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.splitlines()
    unexpected = [line for line in status if not line.endswith('configs/colab.json')]
    assert not unexpected, f'Unexpected checkout changes: {unexpected}'
    if status:
        subprocess.run(['git', 'restore', 'configs/colab.json'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=PROJECT_ROOT, check=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '.', '--no-deps'], returncode=0)

In [19]:
# 2. Restore fixed-Q96 inputs and any completed Task 7 artifacts.
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
input_archive = DRIVE_ARTIFACT_ROOT / 'task2_stagea_bundle.tar.gz'
assert input_archive.is_file(), input_archive
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(input_archive, TASK2_ROOT)
manifest = TASK2_ROOT / 'fixed_q96_manifest.csv'
assert manifest.is_file(), manifest
TASK7_ROOT = ARTIFACT_ROOT / 'task7'
DRIVE_TASK7_ROOT = DRIVE_ARTIFACT_ROOT / 'task7'
if DRIVE_TASK7_ROOT.is_dir():
    shutil.copytree(DRIVE_TASK7_ROOT, TASK7_ROOT, dirs_exist_ok=True)
print('Task 7 inputs ready')

Task 7 inputs ready


In [20]:
# 3. Extract once. Reuse only an artifact produced by the current extractor configuration.
import json
feature_table = TASK7_ROOT / 'frequency_features.csv'
extraction_report = TASK7_ROOT / 'extraction_report.json'
frequency_config = json.loads((PROJECT_ROOT / 'configs/colab.json').read_text())['frequency']
saved_report = json.loads(extraction_report.read_text()) if extraction_report.is_file() else {}
reuse_features = (
    feature_table.is_file()
    and saved_report.get('configuration') == frequency_config
    and saved_report.get('final_test_read') is False
)
if reuse_features:
    print('SKIP complete frequency extraction')
else:
    subprocess.run([
        sys.executable, 'scripts/extract_frequency_features.py',
        '--manifest', str(manifest),
        '--output', str(feature_table),
        '--report', str(extraction_report),
        '--cache-root', '/content/frequency_feature_cache',
        '--matching-policy', 'fixed_q96',
        '--workers', '4',
    ], cwd=PROJECT_ROOT, check=True)
    DRIVE_TASK7_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(feature_table, DRIVE_TASK7_ROOT / feature_table.name)
    shutil.copy2(extraction_report, DRIVE_TASK7_ROOT / extraction_report.name)
print('Feature table:', feature_table)

Feature table: /content/cya-techjam26/artifacts/task7/frequency_features.csv


In [21]:
# 4. Train both representations across three seeds; sync every completed run.
variants = ('magnitude', 'magnitude_phase')
seeds = (42, 43, 44)
for variant in variants:
    for seed in seeds:
        local_run = TASK7_ROOT / variant / f'seed_{seed}'
        drive_run = DRIVE_TASK7_ROOT / variant / f'seed_{seed}'
        if (local_run / 'report.json').is_file():
            print(f'SKIP complete: {variant}, seed {seed}')
            continue
        print(f'RUN: {variant}, seed {seed}')
        subprocess.run([
            sys.executable, 'scripts/train_frequency_baseline.py',
            '--features', str(feature_table),
            '--output', str(local_run),
            '--variant', variant,
            '--seed', str(seed),
        ], cwd=PROJECT_ROOT, check=True)
        shutil.copytree(local_run, drive_run, dirs_exist_ok=True)

RUN: magnitude, seed 42
RUN: magnitude, seed 43
RUN: magnitude, seed 44
RUN: magnitude_phase, seed 42
RUN: magnitude_phase, seed 43
RUN: magnitude_phase, seed 44


In [22]:
# 5. Paired representation decision. This does not enable the Stage 1 early exit.
comparison_path = TASK7_ROOT / 'variant_comparison.json'
subprocess.run([
    sys.executable, 'scripts/compare_frequency_variants.py',
    '--task7-root', str(TASK7_ROOT),
    '--output', str(comparison_path),
], cwd=PROJECT_ROOT, check=True)
shutil.copy2(comparison_path, DRIVE_TASK7_ROOT / comparison_path.name)
comparison = json.loads(comparison_path.read_text())
print('Selected frequency representation:', comparison['selected_representation'])
print('Early exit enabled:', comparison['stage1_early_exit_enabled'])
print(json.dumps(comparison['aggregate'], indent=2))

Selected frequency representation: magnitude
Early exit enabled: False
{
  "magnitude_accuracy_mean": 0.8303030303030304,
  "magnitude_phase_accuracy_mean": 0.8000000000000002,
  "phase_accuracy_mean_delta": -0.030303030303030276,
  "phase_ai_generated_accuracy_mean_delta": -0.0449438202247191,
  "phase_authentic_accuracy_mean_delta": -0.013157894736842146
}
